In [2]:
import pandas as pd
import os

# Ruta correcta en Windows
df = pd.read_parquet(r"C:\Users\sergi\Documents\Pruebas Técnicas\Nequi DS\Analisis_fraude\analisis-fraude\data\01_raw\sample_data_0006_part_00.parquet")



In [3]:
data= df.sample(n=300, random_state=42)

In [4]:
# Mostrar información general del dataset
data_info = data.info()
data_head = data.head()

# Mostrar una descripción estadística inicial
data_description = data.describe()

data_info, data_head, data_description

<class 'pandas.core.frame.DataFrame'>
Index: 300 entries, 1081054 to 945489
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   merchant_id         300 non-null    object        
 1   _id                 300 non-null    object        
 2   subsidiary          300 non-null    object        
 3   transaction_date    300 non-null    datetime64[ns]
 4   account_number      300 non-null    object        
 5   user_id             300 non-null    object        
 6   transaction_amount  300 non-null    object        
 7   transaction_type    300 non-null    object        
dtypes: datetime64[ns](1), object(7)
memory usage: 21.1+ KB


(None,
                               merchant_id                               _id  \
 1081054  075d178871d8d48502bf1f54887e52fe  b69a8ad1a8280b8e6f0bd38b4cce8a4a   
 2471502  817d18cd3c31e40e9bff0566baae7758  650dcc8f0c905205f35b65d7ac5998b2   
 6868479  817d18cd3c31e40e9bff0566baae7758  87be795748f41e4a4a97d783ce1f14cf   
 9424293  838a8fa992a4aa2fb5a0cf8b15b63755  d99542642ab30f13a7676c46b423ea22   
 6613222  817d18cd3c31e40e9bff0566baae7758  be7fd6155c65aeb95df167bf07dffd90   
 
                                subsidiary    transaction_date  \
 1081054  a8bd47b85e2b946052d816b208b3f0a4 2021-06-28 16:26:07   
 2471502  540c5783008d512771471e186f77d9be 2021-10-14 18:12:15   
 6868479  e3a9434b7e145d952d296cb255055b0f 2021-04-06 15:21:37   
 9424293  207693b4bb0a84615b2a4f002a483b0a 2021-08-16 14:59:33   
 6613222  855e9baef9ba84ed74598c5a7de076d7 2021-09-17 20:39:45   
 
                            account_number                           user_id  \
 1081054  cb8cc7fef9d4410878603da

In [5]:
# Ordenar los datos por usuario y fecha de transacción para análisis temporal
data_sorted = data.sort_values(by=["user_id", "transaction_date"])

# Calcular la diferencia de tiempo entre transacciones consecutivas (en horas)
data_sorted["time_diff"] = data_sorted.groupby("user_id")["transaction_date"].diff().dt.total_seconds() / 3600

# Calcular la suma acumulada de los montos dentro de una ventana móvil de 24 horas
window = pd.Timedelta(hours=24)

data_sorted["rolling_sum"] = (
    data_sorted.groupby("user_id")
    .apply(lambda x: x.set_index("transaction_date")["transaction_amount"].rolling(window=window).sum())
    .reset_index(level=0, drop=True)
)

# Calcular el conteo de transacciones en la misma ventana móvil
data_sorted["rolling_count"] = (
    data_sorted.groupby("user_id")
    .apply(lambda x: x.set_index("transaction_date")["transaction_amount"].rolling(window=window).count())
    .reset_index(level=0, drop=True)
)

# Identificar transacciones sospechosas (e.g., montos acumulados altos y conteos significativos)
threshold_amount = 1000  # Umbral ejemplo para suma acumulada
threshold_count = 3      # Umbral ejemplo para conteo de transacciones

data_sorted["suspicious"] = (
    (data_sorted["rolling_sum"] >= threshold_amount) & 
    (data_sorted["rolling_count"] >= threshold_count)
)

# Resumen de las transacciones sospechosas
suspicious_transactions = data_sorted[data_sorted["suspicious"]]

#import ace_tools as tools; tools.display_dataframe_to_user(name="Transacciones Sospechosas", dataframe=suspicious_transactions)

#data_sorted.head(), suspicious_transactions.head()


C:\Users\sergi\AppData\Local\Temp\ipykernel_26444\199883338.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.set_index("transaction_date")["transaction_amount"].rolling(window=window).sum())
C:\Users\sergi\AppData\Local\Temp\ipykernel_26444\199883338.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.set_index("transaction_date")["transaction_amount"].rolling(wind

In [7]:
data_sorted

,merchant_id,_id,subsidiary,transaction_date,account_number,user_id,transaction_amount,transaction_type,time_diff,rolling_sum,rolling_count,suspicious
1336921,075d178871d8d48502bf1f54887e52fe,c8c1d3a0b2d2b2852ee863ec4d5ef848,ef6d127542ef1cb11b6307f08a2396fd,2021-07-08 17:26:51,4dd2848ba53e8828014545ebb318fd45,02f3d8b4d4d4fded73d66bc96b622588,71.33346014,CREDITO,NaN,NaN,NaN,False
5628872,817d18cd3c31e40e9bff0566baae7758,aba7151fbc4c05e64b82b48a5a2b16ab,32965ddf2ce727cc791ab4293bfdf896,2021-03-29 09:19:49,5c38f01e3c060b16d196d27406202184,035fa28db59eabdaa09dddca20fabb2e,356.66730074,DEBITO,NaN,NaN,NaN,False
9406996,838a8fa992a4aa2fb5a0cf8b15b63755,a2f00f5852c5d2159fd7cff4b6001e1d,8c52a2d7745e37bcee79717300f796e3,2021-01-29 11:24:05,cb3b90c9ae3778a188e02fa4ff84223b,036f8e2fc791002f1a3f9f17ef1ad639,11.88891002,DEBITO,NaN,NaN,NaN,False
3508412,817d18cd3c31e40e9bff0566baae7758,9ed4512df5f217e7208d0a719ea55c34,6f5b35cf1fd563d9b7c99da05159fb8c,2021-08-29 09:33:40,af0b6b723429d9d7ce8bfec41c27d3cf,04152164186eaefe1437135b4e4f6fa8,713.33460148,DEBITO,NaN,NaN,NaN,False
4770224,817d18cd3c31e40e9bff0566baae7758,0926279b80d867de40f9d08a253c0ebc,d79252bc18416e40b8bd703a23d324ec,2021-02-07 19:50:58,010901b62115a585555a187309352132,052f48dc2f5080d3dc57271237f2a397,202.11147042,DEBITO,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
6181084,817d18cd3c31e40e9bff0566baae7758,d17361e16aadc22d740a4e90a9203db8,4930cc4d80cae0d9c1a463ca6245be54,2021-10-11 16:37:06,025da8457d4bc4dc4e21cbe6b6fbecdc,fc94dcf85010d248346bf55fc60814e3,59.44455012,DEBITO,NaN,NaN,NaN,False
1575415,075d178871d8d48502bf1f54887e52fe,bb2e51aef21a769235879265f80b156c,9c06a9614a73a916cc73777cc330bc6c,2021-04-16 17:46:44,a9848588a712d8253f77f851ac4c7a8f,fe8b835a4b9030be1e6768707eaefd04,35.66673007,CREDITO,NaN,NaN,NaN,False
8490510,817d18cd3c31e40e9bff0566baae7758,db6c97b9c3d68b0445daf86a1bd90ec3,11f4240fc4ce6c16d87dff573bbc1c96,2021-11-15 19:05:38,b4a4b64d38ec9613836eed5d6d61d694,fed85850796159cf8af62f42d86542f9,118.88910024,DEBITO,NaN,NaN,NaN,False
3360388,817d18cd3c31e40e9bff0566baae7758,7b5b3e9744377f0331252f4352b3cea9,4baa10d32a94dbfe6e644fd0fb5d4ad5,2021-01-16 11:32:07,6548fbf3dc9558cff20522bf0238a07c,feec95133dc2f0ce44cca3a772c0c006,71.33346014,DEBITO,NaN,NaN,NaN,False
